In [2]:
# Example path: output_final/prompt/P1_system1_target2_6.0_1.json
# Read all json files in output_final/prompt/
# Extract system, target, region_bboxes
# Calculate number of region_bboxes
# Compare by system


import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.anova import AnovaRM
import warnings
warnings.filterwarnings("ignore")
import itertools
from collections import defaultdict
from glob import glob
from IPython.display import display, HTML


PATH = "output_final/prompt/"
data_list = []

for f in glob(os.path.join(PATH, "*.json")):
    with open(f, "r") as infile:
        data = json.load(infile)
        # Extract system, target, region_bboxes
        filename = os.path.basename(f)
        parts = filename.split("_")
        if len(parts) < 4:
            continue
        data['participant'] = parts[0]
        data['system'] = parts[1]
        data['target'] = parts[2]
        data_list.append(data)
df = pd.DataFrame(data_list)
df['num_regions'] = df['region_bboxes'].apply(len)

In [7]:
import numpy as np
import pandas as pd
from scipy import stats

# wide 형식으로 변환 (참가자별 baseline, OOPT 값)
wide = df.pivot_table(index="participant", columns="system", values="num_regions")

# 대응표본 t-test
t, p = stats.ttest_rel(wide["system1"], wide["system2"], nan_policy="omit")

# 기술통계
m_b, sd_b = np.mean(wide["system1"]), np.std(wide["system1"], ddof=1)
m_o, sd_o = np.mean(wide["system2"]), np.std(wide["system2"], ddof=1)

print(f"Baseline: M={m_b:.2f}, SD={sd_b:.2f}")
print(f"OOPT:     M={m_o:.2f}, SD={sd_o:.2f}")
print(f"Paired t-test: t={t:.2f}, p={p}")


Baseline: M=4.50, SD=0.67
OOPT:     M=7.08, SD=1.98
Paired t-test: t=-4.33, p=0.0011882961369123221
